***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

# Plotting
import matplotlib.pyplot as plt 
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'SACOG Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Housing', 'config')

In [ ]:
path_housing = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'SACOG Housing Dataset')


In [ ]:
exec(open(os.path.join(path_config0, 'Functions.py')).read())


***

SACOG Housing Permit Data

***

In [ ]:
year_start = 2001
year_end   = 2023

years_to_import = range(year_start, year_end+1)

list_df = []

for year in tqdm(years_to_import):
    df_year = pd.read_excel(os.path.join(path_housing, 'Region_SACOG_Permit_Data.xlsx'), sheet_name = str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df)
df_housing = df_housing.set_index(['County', 'Jurisdiction', 'Year']).reset_index()
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'
df_housing = df_housing[df_housing['County'] != 'Region']
df_housing['MPO'] = 'SACOG'
df_housing = df_housing.set_index(['MPO', 'County', 'Jurisdiction', 'Year']).reset_index()
df_housing = df_housing.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])
df_housing

# Export 

# housing_path = os.path.join(path_out, "Housing_Data_Production.xlsx")
# df_housing.to_excel(housing_path, index = False)

# print(f"Data frame exported to {housing_path}")

***

Production_1

***

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Production')

In [ ]:
indicator_name = 'Production_1'

print('Organizing indicator Production_1 by Jurisdictions')
df_prod1_a = df_housing.copy()
df_prod1_a = df_prod1_a[['MPO', 'County', 'Jurisdiction', 'Year', 'Total']]
display(df_prod1_a.head(5))

print('Organizing indicator Production_1 by Counties')
df_prod1_b = df_housing.copy()
df_prod1_b = df_prod1_b.groupby(['MPO', 'County', 'Year'], as_index = False, sort = False)['Total'].agg('sum')
display(df_prod1_b.head(5))

print('Organizing indicator Production_1 by the entire SACOG Region')
df_prod1_c = df_housing.copy()
df_prod1_c = df_prod1_c.groupby(['MPO', 'Year'], as_index = False, sort = False)['Total'].agg('sum')
display(df_prod1_c.head(5))

# # Export
# df_prod1_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'), sheet_name='Jurisdictions', index = False)
# df_prod1_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'), sheet_name='Counties'     , index = False)
# df_prod1_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'), sheet_name='MPO'          , index = False)

# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod1_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod1_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod1_c.to_excel(writer, index = False, sheet_name = 'MPO')

In [ ]:
df_plot = df_prod1_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year', 'County', 'Jurisdiction'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='Jurisdiction', markers=True)
fig.update_layout(title = 'Total Housing Permits by County/Jurisdiction')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total housing Permits by Jurisdiction_line.html'))
fig.show()

In [ ]:
df_plot = df_prod1_b.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County'])
fig = px.line(df_plot, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Total Housing Permits by County')

df_prod1_b.columns = [col.lower() for col in df_prod1_b.columns]
# df_prod1_b.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_Counties_SACOG_Housing_Permit_DOF.csv'), index = False)
display(df_prod1_b.head())

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total housing Permits by County_line.html'))
fig.show()

In [ ]:
df_plot = df_prod1_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year']) 
fig = px.line(df_plot, x='Year', y='value', markers=True)
fig.update_layout(title = 'Total Housing Permits SACOG')

df_prod1_c.columns = [col.lower() for col in df_prod1_c.columns]
# df_prod1_c.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit.csv'), index = False)
display(df_prod1_c.head())

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total housing Permits SACOG_line.html'))
fig.show()

***

Production_4

***

In [ ]:
# Use pd.read_excel to import the Pop_5 jurisdiction data
# Calculate population growth by year using df_pop_5
# Calculate housing growth by year using df_housing "Total" column
# Merge the two files together by county/jurisdiction
# Roll up to Jurisdiction, County, and MPO levels (3 different data frames)
# Make plots

In [ ]:
indicator_name = 'Production_4'

#Importing Datasets 
pop_5_path = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', 'Pop_5 CA Regions')
df_pop_5 = pd.read_excel(os.path.join(pop_5_path, 'Pop_5_DOF_Jurisdictions.xlsx'), sheet_name='Data')
df_pop_5 = df_pop_5.sort_values(['County', 'Jurisdiction', 'Year'], ascending=[True,True,True])

#pop_5 for MPO
df_pop_5['Population Growth'] = df_pop_5.groupby(['County', 'Jurisdiction'])['Population'].diff()

df_pop_5['County'] = df_pop_5['County'].str.upper() 
df_pop_5['Jurisdiction'] = df_pop_5['Jurisdiction'].str.upper() 

#merging the two files together bby county and jurisdiction
df_merged = pd.merge(df_housing, df_pop_5, on=['MPO', 'County', 'Jurisdiction', 'Year'], how='left')
df_merged = df_merged.rename(columns={'Total': 'Housing Growth'})

#Jurisdiction level
print('Organizing indicator Production_4 by Jurisdictions')
df_prod4_a = df_merged.copy()
df_prod4_a = df_prod4_a[['MPO', 'County', 'Jurisdiction', 'Year', 'Housing Growth', 'Population Growth']]
display(df_prod4_a.head(5))

#County level 
print('Organizing indicator Production_4 by Counties')
df_prod4_b = df_merged.copy()
df_prod4_b = df_prod4_b.groupby(['MPO', 'County', 'Year'], as_index=False, sort=False).agg({'Housing Growth': 'sum', 'Population Growth': 'sum'})
display(df_prod4_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Production_4 by MPO')
df_prod4_c = df_merged.copy()
df_prod4_c = df_prod4_c.groupby(['MPO', 'Year'], as_index=False, sort=False).agg({'Housing Growth': 'sum', 'Population Growth': 'sum'})
display(df_prod4_c.head(5))

# Export

# df_prod4_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing and DOF Population Data.xlsx'), sheet_name = 'Jurisdictions', index=False)
# df_prod4_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing and DOF Population Data.xlsx'), sheet_name = 'Counties'     , index=False)
# df_prod4_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing and DOF Population Data.xlsx'), sheet_name = 'MPO'          , index=False)

# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing and DOF Population Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod4_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing and DOF Population Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod4_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing and DOF Population Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod4_c.to_excel(writer, index = False, sheet_name = 'MPO')


print(f"Data frames exported to {path_out}")

In [ ]:
df_plot = df_prod4_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'County', 'Jurisdiction', 'Year'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash = 'Jurisdiction', facet_col = 'variable', markers=True)
fig.update_layout(title = 'Total Housing/Population Growth by Jurisdiction')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total Housing Growth by Jurisdiction_line.html'))
fig.show()

In [ ]:
df_plot = df_prod4_b.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County']) 
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='variable', markers=True)
fig.update_layout(title = 'Total Housing/Population Growth by County')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total Housing Growth by County_line.html'))
fig.show()

In [ ]:
df_plot = df_prod4_c.copy()

df_plot['Healthy Housing Market Growth'] = df_plot['Population Growth']/2

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
fig = px.line(df_plot[df_plot['variable'] != 'Healthy Housing Market Growth'], x='Year', y='value', color='variable', markers=True)
fig.add_trace(go.Scatter(x=df_plot["Year"], y=df_plot[df_plot['variable'] == 'Healthy Housing Market Growth']['value']
                         , name = 'Healthy Housing Market Growth'
                         , line=go.scatter.Line(color="gray", dash="dot")
                        ))
fig.update_layout(title = 'Total Housing/Population Growth SACOG')

df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
df_plot.columns = [col.lower() for col in df_plot.columns]
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit_DOF.csv'), index = False)
display(df_plot.head())
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Total Housing Growth SACOG_line.html'))
fig.show()

***

Production_6

***

In [ ]:
gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

In [ ]:
indicator_name = "Production_6"

metrics = ['SF_total', 'MF_total', 'MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL', 'SF_SFSL']

#Jurisdiction level
print('Organizing indicator Production_6 by Jurisdictions')
df_prod6_a = gb_jurisdiction[metrics].sum() 
display(df_prod6_a.head(5))

#County level 
print('Organizing indicator Production_6 by Counties')
df_prod6_b = gb_county[metrics].sum()
display(df_prod6_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Production_6 by MPO')
df_prod6_c = gb_mpo[metrics].sum()
display(df_prod6_c.head(5))

# Export

# df_prod6_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'), sheet_name = 'Jurisdictions', index=False)
# df_prod6_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'), sheet_name = 'Counties'     , index=False)
# df_prod6_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'), sheet_name = 'MPO'          , index=False)

# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod6_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod6_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_prod6_c.to_excel(writer, index = False, sheet_name = 'MPO')
    
print(f"Data frames exported to {path_out}")

In [ ]:
df_plota = df_prod6_a.copy()
df_plotb = df_prod6_a.copy()
df_plot1 = df_prod6_a.copy()
df_plot2 = df_prod6_a.copy()
df_plot3 = df_prod6_a.copy()
df_plot4 = df_prod6_a.copy()
df_plot5 = df_prod6_a.copy()


df_plota = pd.melt(df_plota, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plota = df_plota[df_plota['variable'] == 'MF_total']
fig = px.line(df_plota, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_MF Units by Jurisdiction_line.html'))
fig.show()

df_plotb = pd.melt(df_plotb, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plotb = df_plotb[df_plotb['variable'] == 'SF_total']
fig = px.line(df_plotb, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_SF Units by Jurisdiction_line.html'))
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot1 = df_plot1[df_plot1['variable'] == 'MF_2to4']
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF 2 to 4 Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_MF 2 to 4 Units by Jurisdiction_line.html'))
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot2 = df_plot2[df_plot2['variable'] == 'MF_5plus']
fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF 5+ Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_MF 5 Plus Units by Jurisdiction_line.html'))
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot3 = df_plot3[df_plot3['variable'] == 'SF_RR']
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Rural Residential Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_SF RR Units by Jurisdiction_line.html'))
fig.show()

df_plot4 = pd.melt(df_plot4, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot4 = df_plot4[df_plot4['variable'] == 'SF_SFLL']
fig = px.line(df_plot4, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Large Lot Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_SF Large Lot Units by Jurisdiction_line.html'))
fig.show()

df_plot5 = pd.melt(df_plot5, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot5 = df_plot5[df_plot5['variable'] == 'SF_SFSL']
fig = px.line(df_plot5, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Small Lot Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_SF Small Lot Units by Jurisdiction_line.html'))
fig.show()

In [ ]:
df_plota = df_prod6_b.copy()
df_plot1 = df_prod6_b.copy()
df_plot3 = df_prod6_b.copy()


df_plota = pd.melt(df_plota, id_vars = ['Year', 'County'])
df_plota = df_plota[df_plota['variable'].isin(['MF_total', 'SF_total'])]
fig = px.line(df_plota, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units by County')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_MF vs SF Units by County_line.html'))
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_2to4', 'MF_5plus'])]
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF Units by County')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_MF Units by County_line.html'))
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County'])
df_plot3 = df_plot3[df_plot3['variable'].isin(['SF_RR', 'SF_SFLL', 'SF_SFSL'])]
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'SF Units by County')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_SF Units by County_line.html'))
fig.show()


In [ ]:

df_plot = df_prod6_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
df_plot.columns = [col.lower() for col in df_plot.columns]
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit.csv'), index = False)
display(df_plot.head())


df_plota = df_prod6_c.copy()
df_plot1 = df_prod6_c.copy()


df_plota = pd.melt(df_plota, id_vars = ['MPO', 'Year'])
df_plota = df_plota[df_plota['variable'].isin(['MF_total', 'SF_total'])]
fig = px.line(df_plota, x='Year', y='value', color='variable', markers=True)
fig.update_layout(title = 'MF vs SF Units (Total) SACOG')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_MF vs SF Units Total SACOG_line.html'))
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL', 'SF_SFSL'])]
fig = px.line(df_plot1, x='Year', y='value', color='variable', markers=True)
fig.update_layout(title = 'MF vs SF Units (Detailed) SACOG')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_MF vs SF Units Detailed SACOG_line.html'))
fig.show()


In [ ]:
df_plot1 = df_prod6_c.copy()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_total', 'SF_SFLL', 'SF_SFSL'])]

df_plot1['percentage'] = 100*df_plot1['value'] / df_plot1.groupby(['MPO', 'Year'])['value'].transform('sum')

df_plot1['variable_sort'] = pd.Categorical(df_plot1['variable'], ['MF_total', 'SF_SFSL', 'SF_SFLL'])
df_plot1 = df_plot1.sort_values(['MPO', 'Year', 'variable_sort'], ascending = [True, False, False])
df_plot1 = df_plot1.drop('variable_sort', axis = 1)

display(df_plot1.head())

fig = px.bar(df_plot1, x='Year', y='percentage', color='variable')
fig.update_layout(title = 'MF vs SF Units (Detailed) SACOG')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_MF vs SF Units Detailed SACOG_line.html'))
# df_plot1.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit_BAR.csv'), index = False)
fig.show()

***

Location_1

***

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Location')

In [ ]:
gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

In [ ]:
indicator_name = "Location_1"

df_housing['COMTYP_AGNL_NA'] = df_housing['COMTYP_AGNL'] + df_housing['COMTYP_NA']

metrics = ['COMTYP_CC', 'COMTYP_EC', 'COMTYP_DC', 'COMTYP_RR', 'COMTYP_AGNL_NA']

#Jurisdiction level
print('Organizing indicator Location_1 by Jurisdictions')
df_loc1_a = gb_jurisdiction[metrics].sum() 
display(df_loc1_a.head(5))

#County level 
print('Organizing indicator Location_1 by Counties')
df_loc1_b = gb_county[metrics].sum()
display(df_loc1_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Location_1 by MPO')
df_loc1_c = gb_mpo[metrics].sum()
display(df_loc1_c.head(5))

# Export

# df_loc1_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'Jurisdictions', index=False)
# df_loc1_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'Counties'     , index=False)
# df_loc1_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'MPO'          , index=False)

# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc1_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc1_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc1_c.to_excel(writer, index = False, sheet_name = 'MPO')

print(f"Data frames exported to {path_out}")



In [ ]:
df_plot1 = df_loc1_a.copy()
df_plot2 = df_loc1_a.copy()
df_plot3 = df_loc1_a.copy()
df_plot4 = df_loc1_a.copy()
df_plot5 = df_loc1_a.copy()


df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot1 = df_plot1[df_plot1['variable'] == 'COMTYP_CC']
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Commercial Corridor Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_COMTYP CC by Jurisdiction_line.html'))
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot2 = df_plot2[df_plot2['variable'] == 'COMTYP_EC']
fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Established Corridor Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_COMTYP EC by Jurisdiction_line.html'))
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot3 = df_plot3[df_plot3['variable'] == 'COMTYP_DC']
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Developing Corridor Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_COMTYP DC by Jurisdiction_line.html'))
fig.show()

df_plot4 = pd.melt(df_plot4, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot4 = df_plot4[df_plot4['variable'] == 'COMTYP_RR']
fig = px.line(df_plot4, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Rural Residential Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_COMTYP RR by Jurisdiction_line.html'))
fig.show()

df_plot5 = pd.melt(df_plot5, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot5 = df_plot5[df_plot5['variable'] == 'COMTYP_AGNL_NA']
fig = px.line(df_plot5, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Agriculture and Other and NA Units by Jurisdiction')
# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_COMTYP AGNL NA by Jurisdiction_line.html'))
fig.show()

In [ ]:
df_plot = df_loc1_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
df_plot.columns = [col.lower() for col in df_plot.columns]
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit.csv'), index = False)
display(df_plot.head())

***

Location_2a

***

In [ ]:
gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

In [ ]:
indicator_name = "Location_2a"

metrics = ['Total', 'GRZ_TOT']

#Jurisdiction level
print('Organizing indicator Location_2a by Jurisdictions')
df_loc2a_a = gb_jurisdiction[metrics].sum()
df_loc2a_a['GRZ_SHR'] = 100*df_loc2a_a['GRZ_TOT']/df_loc2a_a['Total']
df_loc2a_a = df_loc2a_a.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_a.head(5))

#County level 
print('Organizing indicator Location_2a by Counties')
df_loc2a_b = gb_county[metrics].sum()
df_loc2a_b['GRZ_SHR'] = 100*df_loc2a_b['GRZ_TOT']/df_loc2a_b['Total']
df_loc2a_b = df_loc2a_b.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_b.head(5))

#MPO level
#Jurisdiction level
print('Organizing indicator Location_2a by MPO')
df_loc2a_c = gb_mpo[metrics].sum()
df_loc2a_c['GRZ_SHR'] = 100*df_loc2a_c['GRZ_TOT']/df_loc2a_c['Total']
df_loc2a_c = df_loc2a_c.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_c.head(5))

# Export

# df_loc2a_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'Jurisdictions', index=False)
# df_loc2a_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'Counties'     , index=False)
# df_loc2a_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),sheet_name = 'MPO'          , index=False)


# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2a_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2a_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2a_c.to_excel(writer, index = False, sheet_name = 'MPO')


In [ ]:
df_plot = df_loc2a_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County', 'Jurisdiction'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='Jurisdiction', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion by Jurisdiction')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Green Zone Prop by Jurisdiction_line.html'))
fig.show()

In [ ]:
df_plot = df_loc2a_b.copy()


df_loc2a_b.columns = [col.lower() for col in df_prod1_b.columns]
# df_loc2a_b.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_Counties_SACOG_Housing_Permit.csv'), index = False)
display(df_loc2a_b.head())


df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County'])
fig = px.line(df_plot, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion by County')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Green Zone Prop by County_line.html'))
fig.show()

In [ ]:
df_plot = df_loc2a_c.copy()


df_loc2a_c.columns = [col.lower() for col in df_loc2a_c.columns]
# df_loc2a_c.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit.csv'), index = False)
display(df_loc2a_c.head())


df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year']) 
fig = px.line(df_plot, x='Year', y='value', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Green Zone Prop SACOG_line.html'))
fig.show()

***

Location_2b

***

In [ ]:
gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

In [ ]:
indicator_name = "Location_2b"

df_housing['GRZ_MF'   ] = df_housing['GRZ_MF2t4'   ] + df_housing['GRZ_MF5'   ]
df_housing['GRZ_MF_NO'] = df_housing['GRZ_MF2t4_NO'] + df_housing['GRZ_MF5_NO']

metrics = ['GRZ_SF', 'GRZ_MF', 'GRZ_MF2t4', 'GRZ_MF5', 'GRZ_SFLL', 'GRZ_SFSL', 'GRZ_SFSL_NO', 'GRZ_RR_SFLL_NO', 'GRZ_MF_NO']


#Jurisdiction level for Green Zone region 
print('Organizing indicator Location_2a by Jurisdictions for Green Zone region')
df_loc2b_a = gb_jurisdiction[metrics].sum()
display(df_loc2b_a.head(5))

#County level for Green Zone Region
print('Organizing indicator Location_2a by Counties for Green Zone region')
df_loc2b_b = gb_county[metrics].sum()
display(df_loc2b_b.head(5))

#MPO level for Green Zone Region 
print('Organizing indicator Location_2a by MPO for Green Zone region')
df_loc2b_c = gb_mpo[metrics].sum()
display(df_loc2b_c.head(5))



# Export

# df_loc2b_a.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'), index=False)
# df_loc2b_b.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'), index=False)
# df_loc2b_c.to_excel(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'), index=False)


# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Jurisdictions' + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2b_a.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' Counties'      + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2b_b.to_excel(writer, index = False, sheet_name = 'Counties')
# with pd.ExcelWriter(os.path.join(path_out, indicator_name, indicator_name + ' MPO'           + ' SACOG Housing Permit Data.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#     df_loc2b_c.to_excel(writer, index = False, sheet_name = 'MPO')


In [ ]:
df_plot = df_loc2b_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
df_plot.columns = [col.lower() for col in df_plot.columns]
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_SACOG_Housing_Permit.csv'), index = False)
display(df_plot.head())

***

Policy_5

***

In [ ]:
indicator_name = 'Policy_5'
path_proj = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Policy and Planning')

In [ ]:
df_proj = pd.read_excel(os.path.join(path_proj, 'projection (for policy_5).xlsx'))
display(df_proj.head())
display(df_housing.head())

In [ ]:

df_policy5 = df_prod6_c.copy()

metrics = ['SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total']

df_policy5 = df_prod6_c[['MPO', 'Year'] + metrics]
df_policy5 = df_policy5.sort_values(['MPO', 'Year'])

conditions = [   
         df_policy5['Year'].isin(sequence(2000, 2007, 1))
       , df_policy5['Year'].isin(sequence(2008, 2011, 1))
       , df_policy5['Year'].isin(sequence(2012, 2015, 1))
       , df_policy5['Year'].isin(sequence(2016, 2019, 1))
       , df_policy5['Year'].isin(sequence(2020, 2023, 1))
             ]
choices = ["2001-2007", "2008-2011", "2012-2015", "2016-2019", "2020-2023"]
df_policy5["Period"] = np.select(conditions, choices)

display(df_policy5.head())

In [ ]:

df_plot = df_policy5.copy()

df_plot = df_plot.groupby(['MPO', 'Period'], as_index = False)[metrics].mean()
df_plot[metrics] = df_plot[metrics].apply(round)
list_data = [['SACOG', '2024-2040', 110, 600, 2100, 5700]]
df_forecast = pd.DataFrame(list_data, columns = ['MPO', 'Period', 'SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total'])
df_plot = pd.concat([df_plot, df_forecast])
df_plot = df_plot.rename(columns = {'SF_RR':'Rural Residential', 'SF_SFLL':'Single Family-Large Lot'
                                    , 'SF_SFSL':'Single Family-Small Lot', 'MF_total':'Attached'})
# df_plot.to_excel(os.path.join(path_proj, indicator_name, indicator_name + ' MPO' + ' SACOG Housing Permit Data_Time Periods.xlsx'), index=False)
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Period'])
df_plot['variable_sort'] = pd.Categorical(df_plot['variable'], ['Rural Residential', 'Single Family-Large Lot', 'Single Family-Small Lot', 'Attached'])
df_plot = df_plot.sort_values(['MPO', 'Period', 'variable_sort'], ascending = [True, True, False])
df_plot = df_plot.drop('variable_sort', axis = 1)

fig = px.bar(df_plot, x = 'Period', y = 'value', color = 'variable')
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_SACOG_Housing_Permit_Data.csv'), index = False)
# fig.write_html(os.path.join(path_proj, indicator_name, 'plots', indicator_name + ' Forecasted Housing Growth SACOG_bar.html'))

fig.show()

In [ ]:
list_df_forecast = []

for year in range(2024, 2040):

    annual_growth = [110, 600, 2100, 5700]
    
    list_data = [['SACOG', '2024-2040', year] + annual_growth]
    df_forecast = pd.DataFrame(list_data, columns = ['MPO', 'Period', 'Year', 'SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total'])

    list_df_forecast.append(df_forecast)

df_forecast = pd.concat(list_df_forecast)
df_policy5 = pd.concat([df_policy5, df_forecast])
df_policy5 = df_policy5.set_index(['MPO', 'Period', 'Year']).reset_index()

# df_policy5.to_excel(os.path.join(path_proj, indicator_name, indicator_name + ' MPO' + ' SACOG Housing Permit Data.xlsx'), index=False)


In [ ]:
# steps = 3
# housing_2023 = list(df_policy5[df_policy5['Year'] == 2023][metrics].values[0])
# annual_growth = [110, 600, 2100, 5700]
# annual_growth = [home_forecast*steps for home_forecast in annual_growth]
# housing_forecast = [sum(x) for x in zip(housing_2023, annual_growth)]
# housing_forecast

In [ ]:
# ## FIRST ATTEMPT ##

# path_proj = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Policy and Planning')

# df_proj = pd.read_excel(os.path.join(path_proj, 'projection (for policy_5).xlsx'))
# display(df_proj.head())

# # forecast

# forecast_1 = sequence(2001, 2007, 1)
# forecast_2 = sequence(2008, 2011, 1)
# forecast_3 = sequence(2012, 2015, 1)
# forecast_4 = sequence(2016, 2019, 1)
# forecast_5 = sequence(2020, 2024, 1)

# forecasts = [forecast_1, forecast_2, forecast_3, forecast_4, forecast_5]
# housing_type = list(df_proj['housing_type'].unique())
# housing_type.remove('SF_A')

# dict_forecasts = {}


# for forecast in forecasts:
#     dict_housing = {}
#     key = ''.join([str(np.min(forecast)), '-', str( np.max(forecast))])
#     for home in housing_type:
#         dict_years = {}
#         years_forecasted = 0
#         for year in forecast:
#             annual_growth = df_proj[df_proj['housing_type'] == home]['Annual Projection to 2040 (from 2020 MTP/SCS)'].values[0]
#             housing_growth = df_prod6_c[df_prod6_c['Year'] == np.min(forecast)][home].values[0] + annual_growth*years_forecasted
#             dict_years[year] = housing_growth
#             years_forecasted = years_forecasted + 1
#         dict_housing[home] = dict_years
#     dict_forecasts[key] = dict_housing

# print(dict_forecasts)



# list_df_forecasts = []

# for forecast in forecasts:
#     key = ''.join([str(np.min(forecast)), '-', str( np.max(forecast))])
#     df_forecast = pd.DataFrame.from_dict(dict_forecasts[key]).reset_index(names = 'Year')
#     df_forecast = pd.melt(df_forecast, id_vars = 'Year', var_name = 'Housing Type', value_name = 'Forecast')
#     df_forecast['Time Period'] = key
#     list_df_forecasts.append(df_forecast)

# df_forecasts = pd.concat(list_df_forecasts)
# df_forecasts = df_forecasts.set_index('Time Period').reset_index()

# display(df_forecasts.head())



# df_policy5 = df_prod6_c.copy()

# df_policy5 = pd.melt(df_prod6_c, id_vars = ['MPO', 'Year'], var_name = 'Housing Type', value_name = 'Actual Permits')
# df_policy5 = df_policy5[df_policy5['Housing Type'].isin(housing_type)]
# df_policy5 = df_policy5.merge(df_forecasts, on = ['Year', 'Housing Type'])

# df_policy5 = pd.melt(df_policy5, id_vars = ['MPO', 'Time Period', 'Year', 'Housing Type'])
# df_policy5 = df_policy5.sort_values(['MPO', 'Year', 'Housing Type', 'variable'])

# display(df_policy5.head())




# df_plot = df_policy5.copy()

# fig = px.line(df_plot, x='Year', y='value', color='Housing Type', line_dash = 'variable', markers=True, facet_col = 'Time Period')
# for k in fig.layout:
#     if re.search('yaxis[1-9]+', k):
#         fig.layout[k].update(matches=None)
# for k in fig.layout:
#     if re.search('xaxis[1-9]+', k):
#         fig.layout[k].update(matches=None)
        
# fig.update_layout(title = 'Actual Housing Permits vs Forecasted Growth by Time Period SACOG')

# # fig.write_html(os.path.join(path_proj, indicator_name, 'plots', indicator_name + ' Forecasted Housing Growth SACOG_line.html'))

# fig.show()




# df_policy5 = df_policy5.pivot_table(index = ['MPO', 'Time Period', 'Year', 'Housing Type']
#                                     , columns = 'variable'
#                                     , values = 'value').reset_index()

# display(df_policy5.head())



# # Export 
# # df_policy5.to_excel(os.path.join(path_proj, indicator_name, indicator_name + ' MPO' + ' SACOG Housing Permit Data.xlsx'), index=False)


***

Cost_4

***

In [ ]:
#Concat_HCD_Data

year_start = 2018
year_end   = 2023

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(os.path.join(path_housing, 'HCD_Summarized_Yearly.xlsx'), sheet_name = str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df, ignore_index=True)
df_housing = df_housing[~df_housing['JURS_NAME'].str.contains('https')]
df_housing.loc[df_housing['JURS_NAME'].str.contains('COUNTY'), 'JURS_NAME'] = 'UNINCORPORATED'

df_housing

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost')

In [ ]:
gb_jurisdiction = df_housing.groupby(['CNTY_NAME', 'JURS_NAME', 'Year'], as_index=False)
gb_county       = df_housing.groupby(['CNTY_NAME',              'Year'], as_index=False)
gb_mpo          = df_housing.groupby([                          'Year'], as_index=False)

In [ ]:
indicator_name = 'Cost_4'

metrics = ['Total', 'CO_VLI', 'CO_LI', 'CO_MI', 'CO_AMI']

#Jurisdiction level
print('Organizing indicator Cost_4 by Jurisdictions')
df_cost4_a = gb_jurisdiction[metrics].sum() 
display(df_cost4_a.head(5))

#County level 
print('Organizing indicator Cost_4 by Counties')
df_cost4_b = gb_county[metrics].sum()
display(df_cost4_b.head(5))

#MPO level 
print('Organizing indicator Cost_4 by MPO')
df_cost4_c = gb_mpo[metrics].sum()
display(df_cost4_c.head(5))

# # Export
# df_cost4_a.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' Jurisdictions' + '_SACOG HCD Summarized Data.xlsx'), index=False)
# df_cost4_b.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' Counties'      + '_SACOG HCD Summarized Data.xlsx'), index=False)
# df_cost4_c.to_excel(os.path.join(path_out, indicator_name + ' RHNA Income', indicator_name + ' MPO'           + '_SACOG HCD Summarized Data.xlsx'), index=False)

# print(f"Data frames exported to {path_out}")

In [ ]:
df_plot = df_cost4_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year']) 
df_plot = df_plot[df_plot['variable'] != 'Total']
df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['year'])['value'].transform('sum')
df_plot = df_plot.sort_values(['year', 'variable'], ascending = [False, True])
# df_plot.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MPO_HCD_Summarized_Data.csv'), index = False)
display(df_plot.head())


fig = px.line(df_plot, x='year', y='percentage', color='variable', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

# fig.write_html(os.path.join(path_out, indicator_name, 'plots', indicator_name + '_Green Zone Prop SACOG_line.html'))
fig.show()